<a href="https://colab.research.google.com/github/malyadri6761/RAG/blob/master/rag_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.7 MB/s eta 0:00:00


In [2]:
from pypdf import PdfReader

In [3]:
path = "/content/pytorch.pdf"
reader = PdfReader(path)

---

**Extract text page by page**

---

In [4]:
documents = []

for page_number, page in enumerate(reader.pages):
  text = page.extract_text()
  if text:
    documents.append({
        "text" : text,
        "page":page_number

    })
print(len(documents))

268


---

**Create Chunks**

---


In [6]:
def create_chunks(text,chunk_size = 50,overlap=10):
  words = text.split()
  chunks = []

  start = 0
  while start<len(words):
    end = start+chunk_size
    chunk = " ".join(words[start:end])
    chunks.append(chunk)
    start+=chunk_size-overlap
  return chunks

---

**Create Pdf Chunks**

---

In [7]:
all_chunks = []
for document in documents:
  chunks = create_chunks(document['text'])

  for chunk in chunks:
    all_chunks.append({
        'text' : chunk,
        'page' : document['page']
    })
print(len(all_chunks))


1916


In [8]:
for i,chunk in enumerate(all_chunks):
  print(f'chunk : {i}')
  print(f'page : {chunk['page']}')
  print(f'text : {chunk['text']}')
  print('-'*50)

chunk : 0
page : 1
text : Programming PyTorch for Deep Learning Creating and Deploying Deep Learning Applications Ian Pointer
--------------------------------------------------
chunk : 1
page : 2
text : Programming PyTorch for Deep Learning by Ian Pointer Copyright © 2019 Ian Pointer. All rights reserved. Printed in the United States of America. Published by O’Reilly Media, Inc. , 1005 Gravenstein Highway North, Sebastopol, CA 95472. O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions
--------------------------------------------------
chunk : 2
page : 2
text : purchased for educational, business, or sales promotional use. Online editions are also available for most titles ( http://oreilly.com ). For more information, contact our corporate/institutional sales department: 800-998- 9938 or corporate@oreilly.com . Development Editor: Melissa Potter Acquisitions Editor: Jonathan Hassell Production Editor: Katherine Tozer Copyeditor: Sharon Wi

---

**Create Embeddings**

---


In [10]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
texts = [chunk['text'] for chunk in all_chunks]

embeddings = embedding_model.encode(texts,convert_to_numpy=True)

---

**FAISS Index**

---

In [14]:
!pip install faiss-cpu
import faiss
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)
print(index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.7 MB/s eta 0:00:00
1916


---

**Create Retrieval Function**

---

In [18]:
def retreival(query,k=3):
  query_embeddings = embedding_model.encode([query],convert_to_numpy=True)

  distances,indices = index.search(query_embeddings,k)

  results = []

  for rank,idx in enumerate(indices[0]):
    results.append({
        'text':all_chunks[idx]['text'],
        'page':all_chunks[idx]['page'],
        'distance':distances[0][rank]
    })
  return results

---

**Test Retrieval**

---


In [19]:
query = "What is deep  learning?"

results = retreival(query)

for result in results:
  print(f'text : {result["text"]}')
  print(f'page : {result["page"]}')
  print(f'distance : {result["distance"]}')
  print('-'*50)

text : What Is Deep Learning Exactly, and Do I Need a PhD to Understand It? Deep learning’s definition often is more confusing than enlightening. A way of defining it is to say that deep learning is a machine learning technique that uses multiple and numerous layers of nonlinear transforms to progressively
page : 5
distance : 0.34924638271331787
--------------------------------------------------
text : years that followed, that error rate got pushed down further and further, to the point that in 2015, the ResNet architecture obtained a result of 3.6%, which beat the average human performance on ImageNet (5%). We had been outclassed. But What Is Deep Learning Exactly, and Do I Need a
page : 5
distance : 0.6162910461425781
--------------------------------------------------
text : Activation Mapping decoder , Introduction to Super-Resolution deep learning, defined , But What Is Deep Learning Exactly, and Do I Need a PhD to Understand It? degrees parameter , Torchvision Transforms deletion

In [22]:
context = "\n\n".join(

    f"[Page: {result['page']}]\n"
    f"{result['text']}"

    for result in results
)

print(context)

[Page: 5]
What Is Deep Learning Exactly, and Do I Need a PhD to Understand It? Deep learning’s definition often is more confusing than enlightening. A way of defining it is to say that deep learning is a machine learning technique that uses multiple and numerous layers of nonlinear transforms to progressively

[Page: 5]
years that followed, that error rate got pushed down further and further, to the point that in 2015, the ResNet architecture obtained a result of 3.6%, which beat the average human performance on ImageNet (5%). We had been outclassed. But What Is Deep Learning Exactly, and Do I Need a

[Page: 241]
Activation Mapping decoder , Introduction to Super-Resolution deep learning, defined , But What Is Deep Learning Exactly, and Do I Need a PhD to Understand It? degrees parameter , Torchvision Transforms deletion, random , Random Deletion DenseNet , Other Architectures Are Available! differential learning rates , Differential Learning Rates DigitalOcean


In [23]:
prompt = f"""
You are a helpful document question-answering assistant.

Answer the question using ONLY the provided context.

If the answer cannot be found in the context,
say:

"I don't know based on the provided document."

Do not use outside knowledge.

Context:
{context}

Question:
{query}

Answer:
"""

In [28]:
# !pip install groq
from groq import Groq
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')
model = Groq(api_key=groq_api_key)
response = model.chat.completions.create(

    model="openai/gpt-oss-120b",

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],

    temperature=0
)

answer = response.choices[0].message.content

In [31]:
print("Answer:")
print(answer)

print("\nSources:")

for result in results:

    print(

        f"Page {result['page']}"
    )

Answer:
Deep learning is a machine‑learning technique that uses multiple and numerous layers of nonlinear transforms to progressively process data.

Sources:
Page 5
Page 5
Page 241
